# Cross-Site and Source-Specific Duplicate Analysis

This notebook estimates repeated observations in the accepted snapshot before salary, demand, or company metrics are calculated. It keeps proven Computrabajo same-platform URL-fragment repetition separate from conservative cross-site duplication. Ten metadata-conflict groups have an explicit user-approved analytical decision; future ambiguous groups remain manual-review candidates and never enter confirmed counts.

The snapshot records accepted rows, not rejected ingestion attempts. Results therefore describe residual repetition in this snapshot, not a historical ingestion duplicate rate.

## 1. Reproducibility contract

The local snapshot, checksum, dimensions, date range, cutoff, and reviewed path identifiers are fixed. Loading is read-only and execution stops if the approved contract changes.

In [12]:
from __future__ import annotations

import hashlib
import re
import unicodedata
from itertools import combinations
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import pandas as pd

In [13]:
SNAPSHOT_RELATIVE_PATH = Path("data/analysis/jobs_27_07_2026.csv")
EXPECTED_SHA256 = "c3412e3315ee4c3df02f9a4f9c0e570747518f9e1ad70f9a6e1e86ceb260fd71"
EXPECTED_ROWS = 26_619
EXPECTED_COLUMNS = 26
SNAPSHOT_CUTOFF = pd.Timestamp("2026-07-27 23:59:59.999999999", tz="UTC")
EXPECTED_DATE_START = pd.Timestamp("2026-03-11", tz="UTC")
EXPECTED_DATE_END = pd.Timestamp("2026-07-27", tz="UTC")
EXPECTED_SCHEMA = [
    "id", "site", "job_url", "title", "company", "raw_location",
    "country", "state", "career", "scraped_by", "industry_niche",
    "is_premium", "job_type", "salary_min", "salary_max", "currency",
    "salary_source", "description", "cleaned_description", "keywords_json",
    "embedding", "date_posted", "date_scraped", "last_updated",
    "status", "country_code",
]
STRING_COLUMNS = [column for column in EXPECTED_SCHEMA if column not in {
    "salary_min", "salary_max", "is_premium"
}]
NA_TOKENS = ["", "NULL", "null", "None", "none", "NaN", "nan"]
MISSING_TEXT_VALUES = {"", "nan", "none", "null", "<na>"}
FUZZY_MIN_TITLE_JACCARD = 0.72
MAX_FUZZY_BLOCK_ROWS = 200
MIN_MEANINGFUL_DESCRIPTION_LENGTH = 50
DESCRIPTION_PREVIEW_MAX_CHARACTERS = 180
SAMPLE_ROWS = 30
REVIEWED_METADATA_CONFLICT_EVIDENCE_CLASS = "confirmed_same_source_reviewed_metadata_conflict"
REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS = frozenset({
    "666F9D957FEB1BB861373E686DCF3405",
    "A699E67A32F41B1161373E686DCF3405",
    "6CE4B1037111A93261373E686DCF3405",
    "C2479C415C8096D261373E686DCF3405",
    "1E2EE54BC2FCD2B461373E686DCF3405",
    "DBFEA1291BDCF25561373E686DCF3405",
    "825E951BDDE77C0461373E686DCF3405",
    "3FAB267D063F3BFA61373E686DCF3405",
    "3E43DAE0BD02734261373E686DCF3405",
    "275A0658863F95AA61373E686DCF3405",
})
REVIEWED_METADATA_CONFLICT_RATIONALE_BY_PATH_IDENTIFIER = {
    "666F9D957FEB1BB861373E686DCF3405": "exact-content company attribution change",
    "A699E67A32F41B1161373E686DCF3405": "placeholder/disclosed company",
    "6CE4B1037111A93261373E686DCF3405": "placeholder/disclosed company",
    "C2479C415C8096D261373E686DCF3405": "spelling/name variant",
    "1E2EE54BC2FCD2B461373E686DCF3405": "placeholder/disclosed company",
    "DBFEA1291BDCF25561373E686DCF3405": "placeholder/disclosed company",
    "825E951BDDE77C0461373E686DCF3405": "placeholder/disclosed company",
    "3FAB267D063F3BFA61373E686DCF3405": "spelling/name variant",
    "3E43DAE0BD02734261373E686DCF3405": "title wording update",
    "275A0658863F95AA61373E686DCF3405": "exact-content company attribution change",
}
EXPECTED_REVIEWED_METADATA_CONFLICT_GROUPS = 10
EXPECTED_REVIEWED_METADATA_CONFLICT_ROWS = 78

In [14]:
def locate_repo_root(start: Path) -> Path:
    """Locate the repository root from a notebook or repository directory."""
    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(f"Repository root not found from {resolved_start}.")


def sha256_file(path: Path, chunk_size: int = 1_048_576) -> str:
    """Compute a file SHA-256 digest using bounded memory."""
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


repo_root = locate_repo_root(Path.cwd())
snapshot_path = repo_root / SNAPSHOT_RELATIVE_PATH
if not snapshot_path.is_file():
    raise FileNotFoundError(f"Approved snapshot not found at {snapshot_path}.")
actual_sha256 = sha256_file(snapshot_path)
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError("Snapshot checksum differs from the approved contract.")
jobs = pd.read_csv(
    snapshot_path, encoding="utf-8", sep=",", quotechar='"',
    dtype={column: "string" for column in STRING_COLUMNS},
    keep_default_na=True, na_values=NA_TOKENS, on_bad_lines="error", low_memory=False,
)
if jobs.shape != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise ValueError(f"Expected {(EXPECTED_ROWS, EXPECTED_COLUMNS)}, received {jobs.shape}.")
if jobs.columns.tolist() != EXPECTED_SCHEMA:
    raise ValueError("Snapshot schema or column order differs from the approved contract.")
snapshot_dates = pd.to_datetime(jobs["date_scraped"], errors="coerce", utc=True)
if snapshot_dates.isna().any():
    raise ValueError("Snapshot contains missing or unparseable date_scraped values.")
if (
    snapshot_dates.min().normalize() != EXPECTED_DATE_START
    or snapshot_dates.max().normalize() != EXPECTED_DATE_END
    or snapshot_dates.gt(SNAPSHOT_CUTOFF).any()
):
    raise ValueError("Snapshot date coverage differs from the approved contract.")
pd.Series({
    "snapshot": str(SNAPSHOT_RELATIVE_PATH), "sha256": actual_sha256,
    "rows": len(jobs), "columns": len(jobs.columns),
    "coverage": f"{snapshot_dates.min().date()} through {snapshot_dates.max().date()}",
}, name="reproducibility_contract")

snapshot                    data/analysis/jobs_27_07_2026.csv
sha256      c3412e3315ee4c3df02f9a4f9c0e570747518f9e1ad70f...
rows                                                    26619
columns                                                    26
coverage                        2026-03-11 through 2026-07-27
Name: reproducibility_contract, dtype: object

## 2. Conservative normalization and exact-key audit

Normalization supports comparison while raw fields remain unchanged. URL cleanup removes fragments and recognized tracking parameters but retains unknown query parameters. Exact raw `job_url` duplicates, canonical collisions, same-source path evidence, confirmed cross-site edges, and fuzzy pairs are separate concepts.

In [15]:
COMPANY_PLACEHOLDERS = {
    "unknown", "confidential", "confidential company", "empresa confidencial",
    "importante empresa del sector", "empresa del sector", "n a", "na",
    "not available", "sin nombre", "company",
}
LEGAL_SUFFIX_PATTERNS = [
    ("s", "a", "de", "c", "v"), ("s", "de", "r", "l", "de", "c", "v"),
    ("sapi", "de", "cv"), ("sa", "de", "cv"), ("s", "c"),
    ("incorporated",), ("corporation",), ("limited",), ("inc",),
    ("corp",), ("llc",), ("ltd",), ("sa",), ("cv",),
]
GENERIC_TITLES = {"developer", "engineer", "analyst", "manager"}
TRACKING_PARAMETERS = {"fbclid", "gclid", "mc_cid", "mc_eid", "ref", "referrer", "source", "trackingid", "trk"}
COMPUTRABAJO_FRAGMENT_PATTERN = re.compile(r"^lc=ListOffers-Score\d+-\d+$", re.IGNORECASE)
COMPUTRABAJO_PATH_ID_PATTERN = re.compile(r"([0-9a-f]{32})$", re.IGNORECASE)
ANALYTICAL_FIELDS = [
    "cleaned_description", "keywords_json", "embedding", "career",
    "industry_niche", "job_type", "salary_min", "salary_max",
    "currency", "salary_source", "date_posted", "status",
]


def normalize_text(value: object) -> str:
    """Normalize Unicode, case, accents, punctuation, and whitespace."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.casefold() in MISSING_TEXT_VALUES:
        return ""
    decomposed = unicodedata.normalize("NFKD", text.casefold())
    unaccented = "".join(char for char in decomposed if not unicodedata.combining(char))
    return " ".join(re.sub(r"[^a-z0-9]+", " ", unaccented).split())


def normalize_company(value: object) -> str:
    """Normalize a company and conservatively remove one trailing legal suffix."""
    tokens = normalize_text(value).split()
    for suffix in sorted(LEGAL_SUFFIX_PATTERNS, key=len, reverse=True):
        if tuple(tokens[-len(suffix):]) == suffix:
            tokens = tokens[:-len(suffix)]
            break
    return " ".join(tokens)


def canonicalize_url(value: object) -> tuple[str, bool]:
    """Remove fragments and known tracking parameters while retaining unknown parameters."""
    if pd.isna(value) or str(value).strip().casefold() in MISSING_TEXT_VALUES:
        return "", False
    raw_url = str(value).strip()
    parse_target = raw_url if "://" in raw_url else f"//{raw_url}"
    try:
        parsed = urlsplit(parse_target)
        host = parsed.hostname.casefold() if parsed.hostname else ""
        if not host:
            return raw_url, True
        port = f":{parsed.port}" if parsed.port else ""
    except ValueError:
        return raw_url, True
    kept_query = sorted(
        (key, item) for key, item in parse_qsl(parsed.query, keep_blank_values=True)
        if key.casefold() not in TRACKING_PARAMETERS and not key.casefold().startswith("utm_")
    )
    path = re.sub(r"/{2,}", "/", parsed.path).rstrip("/") or "/"
    scheme = parsed.scheme.casefold() if parsed.scheme else "https"
    return urlunsplit((scheme, f"{host.removeprefix('www.')}{port}", path, urlencode(kept_query), "")), False


def exact_text_hash(value: str) -> str:
    """Hash non-empty normalized text."""
    return hashlib.sha256(value.encode("utf-8")).hexdigest() if value else ""


url_results = jobs["job_url"].map(canonicalize_url)
canonical_urls = url_results.map(lambda result: result[0]).astype("string")
malformed_urls = url_results.map(lambda result: result[1]).astype(bool)
description_norm = jobs["description"].map(normalize_text).astype("string")
analysis_jobs = jobs.assign(
    row_number=pd.RangeIndex(len(jobs)), site_norm=jobs["site"].map(normalize_text).astype("string"),
    title_norm=jobs["title"].map(normalize_text).astype("string"),
    company_norm=jobs["company"].map(normalize_company).astype("string"),
    location_norm=jobs["raw_location"].map(normalize_text).astype("string"),
    canonical_url=canonical_urls, canonical_evidence_key=canonical_urls.mask(malformed_urls, ""),
    malformed_url=malformed_urls, description_norm=description_norm,
    description_hash=description_norm.map(exact_text_hash).astype("string"),
    date_scraped_parsed=pd.to_datetime(jobs["date_scraped"], errors="coerce", utc=True),
)
analysis_jobs = analysis_jobs.assign(
    company_placeholder=analysis_jobs["company_norm"].isin(COMPANY_PLACEHOLDERS),
    generic_title=analysis_jobs["title_norm"].isin(GENERIC_TITLES),
)
key_audit = pd.DataFrame([{
    "key": key, "empty_rows": int(analysis_jobs[key].map(normalize_text).eq("").sum()),
    "duplicate_rows_involved": int(analysis_jobs[key].duplicated(keep=False).sum()),
    "excess_copies": int(analysis_jobs[key].duplicated(keep="first").sum()),
    "denominator_rows": len(analysis_jobs),
} for key in ["id", "job_url"]])
canonical_present = analysis_jobs["canonical_url"].ne("") & ~analysis_jobs["malformed_url"]
canonical_sizes = analysis_jobs.loc[canonical_present].groupby("canonical_url", sort=True).size()
repeated_canonical_urls = set(canonical_sizes.loc[canonical_sizes.gt(1)].index)
canonical_collision_rows = analysis_jobs.loc[analysis_jobs["canonical_url"].isin(repeated_canonical_urls)].copy()
canonical_collision_groups = canonical_collision_rows.groupby("canonical_url", sort=True).agg(
    rows_involved=("row_number", "size"), source_count=("site_norm", "nunique"),
    source_combination=("site_norm", lambda values: " + ".join(sorted(set(values)))),
).reset_index()
canonical_collision_groups["additional_collisions"] = canonical_collision_groups["rows_involved"] - 1
canonical_url_audit = pd.Series({
    "eligible_url_rows": int(canonical_present.sum()),
    "malformed_or_empty_url_rows": int((~canonical_present).sum()),
    "collision_groups": len(canonical_collision_groups),
    "collision_rows_involved": len(canonical_collision_rows),
    "additional_collisions": int(canonical_collision_groups["additional_collisions"].sum()),
    "cross_site_collision_groups": int(canonical_collision_groups["source_count"].gt(1).sum()),
}, name="canonical_url_audit")
{"exact_key_audit": key_audit, "canonical_url_audit": canonical_url_audit}

{'exact_key_audit':        key  empty_rows  duplicate_rows_involved  excess_copies  \
 0       id           0                        0              0   
 1  job_url           0                        0              0   
 
    denominator_rows  
 0             26619  
 1             26619  ,
 'canonical_url_audit': eligible_url_rows              26619
 malformed_or_empty_url_rows        0
 collision_groups                1361
 collision_rows_involved         8518
 additional_collisions           7157
 cross_site_collision_groups        0
 Name: canonical_url_audit, dtype: int64}

## 3. Computrabajo stable listing identity

The repeated Computrabajo rows share the exact vacancy path and differ only in `#lc=ListOffers-Score...` fragments. Automatic exact-content and strong-path rules remain unchanged. The ten approved metadata-conflict path identifiers are promoted only after they still pass the same fragment/path integrity checks and are still members of the expected automatic metadata-conflict cohort.

In [16]:
def computrabajo_url_evidence(value: object) -> dict[str, object]:
    """Extract fragment-free source identity and reviewable URL evidence."""
    raw_url = "" if pd.isna(value) else str(value).strip()
    parsed = urlsplit(raw_url)
    path = re.sub(r"/{2,}", "/", parsed.path).rstrip("/") or "/"
    host = (parsed.hostname or "").casefold().removeprefix("www.")
    fragment_free_url = urlunsplit((parsed.scheme.casefold(), parsed.netloc.casefold(), path, parsed.query, ""))
    match = COMPUTRABAJO_PATH_ID_PATTERN.search(path.rsplit("/", 1)[-1])
    return {
        "source_listing_identity": f"computrabajo:{host}{path}", "canonical_path": path,
        "path_identifier": match.group(1).upper() if match else "",
        "path_identifier_reliably_present": bool(match), "fragment_free_raw_url": fragment_free_url,
        "raw_fragment": parsed.fragment,
        "expected_tracking_fragment": bool(COMPUTRABAJO_FRAGMENT_PATTERN.fullmatch(parsed.fragment)),
    }


def nonempty_unique_count(values: pd.Series) -> int:
    """Count distinct non-empty values."""
    return int(values.loc[values.ne("")].nunique())


def classify_computrabajo_group(row: pd.Series) -> str:
    """Classify source-path evidence without claiming an undocumented contract."""
    fragment_only = (
        row["fragment_free_url_count"] == 1
        and row["expected_fragment_rows"] == row["times_seen"]
        and row["unique_fragments"] > 1
    )
    same_context = row["unique_titles"] == 1 and row["unique_companies"] == 1
    exact_complete_text = (
        row["unique_nonempty_descriptions"] == 1
        and row["descriptions_present"] == row["times_seen"]
    )
    if fragment_only and same_context and exact_complete_text:
        return "confirmed_same_source_exact_content"
    if fragment_only and same_context:
        return "confirmed_same_source_strong_path_evidence"
    return "ambiguous_or_conflicting_source_observations"


computrabajo_mask = analysis_jobs["site_norm"].eq("computrabajo") & ~analysis_jobs["malformed_url"]
computrabajo_rows = analysis_jobs.loc[computrabajo_mask].copy()
url_evidence = pd.DataFrame(computrabajo_rows["job_url"].map(computrabajo_url_evidence).tolist(), index=computrabajo_rows.index)
computrabajo_rows = computrabajo_rows.join(url_evidence)
identity_sizes = computrabajo_rows.groupby("source_listing_identity", sort=True).size()
repeated_identities = set(identity_sizes.loc[identity_sizes.gt(1)].index)
computrabajo_repeated_rows = computrabajo_rows.loc[
    computrabajo_rows["source_listing_identity"].isin(repeated_identities)
].copy()
populated_analytical = pd.concat(
    [computrabajo_repeated_rows[column].map(normalize_text).ne("") for column in ANALYTICAL_FIELDS], axis=1
).sum(axis=1)
computrabajo_repeated_rows = computrabajo_repeated_rows.assign(
    meaningful_description=computrabajo_repeated_rows["description_norm"].str.len().ge(MIN_MEANINGFUL_DESCRIPTION_LENGTH),
    analytical_fields_populated=populated_analytical,
    description_length=computrabajo_repeated_rows["description_norm"].str.len(),
    stable_id_tiebreaker=computrabajo_repeated_rows["id"].fillna("").astype("string"),
)
representative_ranking = computrabajo_repeated_rows.sort_values(
    ["source_listing_identity", "meaningful_description", "analytical_fields_populated",
     "description_length", "date_scraped_parsed", "stable_id_tiebreaker", "row_number"],
    ascending=[True, False, False, False, True, True, True], kind="mergesort",
)
representatives = representative_ranking.drop_duplicates("source_listing_identity").set_index("source_listing_identity")
computrabajo_listing_level = computrabajo_repeated_rows.groupby("source_listing_identity", sort=True).agg(
    canonical_path=("canonical_path", "first"), path_identifier=("path_identifier", "first"),
    path_identifier_reliably_present=("path_identifier_reliably_present", "all"),
    times_seen=("row_number", "size"), first_seen=("date_scraped_parsed", "min"),
    last_seen=("date_scraped_parsed", "max"), fragment_free_url_count=("fragment_free_raw_url", "nunique"),
    unique_fragments=("raw_fragment", "nunique"), expected_fragment_rows=("expected_tracking_fragment", "sum"),
    unique_titles=("title_norm", "nunique"), unique_companies=("company_norm", "nunique"),
    unique_nonempty_descriptions=("description_hash", nonempty_unique_count),
    descriptions_present=("description_hash", lambda values: int(values.ne("").sum())),
).reset_index()
computrabajo_listing_level["automatic_evidence_class"] = computrabajo_listing_level.apply(
    classify_computrabajo_group, axis=1
)
computrabajo_listing_level = computrabajo_listing_level.join(
    representatives[["id", "row_number"]].rename(columns={
        "id": "representative_row_id", "row_number": "representative_row_number",
    }), on="source_listing_identity",
)
computrabajo_listing_level["additional_observations"] = computrabajo_listing_level["times_seen"] - 1
approved_groups = computrabajo_listing_level.loc[
    computrabajo_listing_level["path_identifier"].isin(REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS)
].copy()
approved_identifiers_found = set(approved_groups["path_identifier"])
if approved_identifiers_found != REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS:
    missing = sorted(REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS - approved_identifiers_found)
    unexpected = sorted(approved_identifiers_found - REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS)
    raise ValueError(f"Reviewed metadata-conflict path identifier mismatch. Missing={missing}; unexpected={unexpected}.")
if not approved_groups["path_identifier"].is_unique:
    raise ValueError("A reviewed path identifier is duplicated across source listing identities.")
if not approved_groups["automatic_evidence_class"].eq("ambiguous_or_conflicting_source_observations").all():
    raise ValueError("A reviewed path identifier is no longer in the expected automatic metadata-conflict cohort.")
approved_group_integrity = (
    approved_groups["path_identifier_reliably_present"]
    & approved_groups["fragment_free_url_count"].eq(1)
    & approved_groups["expected_fragment_rows"].eq(approved_groups["times_seen"])
    & approved_groups["unique_fragments"].gt(1)
    & approved_groups.apply(lambda row: str(row["canonical_path"]).upper().endswith(row["path_identifier"]), axis=1)
)
if not approved_group_integrity.all():
    failed_ids = approved_groups.loc[~approved_group_integrity, "path_identifier"].tolist()
    raise ValueError(f"Reviewed groups lack expected path or tracking-fragment evidence: {failed_ids}.")
approved_underlying_rows = computrabajo_repeated_rows.loc[
    computrabajo_repeated_rows["path_identifier"].isin(REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS)
]
approved_identity_counts = approved_underlying_rows.groupby("path_identifier", sort=True)[
    "source_listing_identity"
].nunique()
if set(approved_identity_counts.index) != REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS or not approved_identity_counts.eq(1).all():
    raise ValueError("Reviewed path identifiers are missing or duplicated across underlying identities.")
if (
    not approved_underlying_rows["expected_tracking_fragment"].all()
    or not approved_underlying_rows.apply(
        lambda row: str(row["canonical_path"]).upper().endswith(row["path_identifier"]), axis=1
    ).all()
):
    raise ValueError("Reviewed underlying rows lack expected tracking-fragment or path evidence.")
computrabajo_listing_level["evidence_class"] = computrabajo_listing_level["automatic_evidence_class"]
reviewed_promotion_mask = computrabajo_listing_level["path_identifier"].isin(
    REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS
)
computrabajo_listing_level.loc[reviewed_promotion_mask, "evidence_class"] = (
    REVIEWED_METADATA_CONFLICT_EVIDENCE_CLASS
)
reviewed_metadata_conflict_groups = computrabajo_listing_level.loc[reviewed_promotion_mask].copy()
unresolved_ambiguous_groups = computrabajo_listing_level.loc[
    computrabajo_listing_level["evidence_class"].eq("ambiguous_or_conflicting_source_observations")
].copy()
computrabajo_class_summary = computrabajo_listing_level.groupby("evidence_class", sort=True).agg(
    listing_identities=("source_listing_identity", "size"), rows_involved=("times_seen", "sum"),
    additional_observations=("additional_observations", "sum"),
).reset_index()
computrabajo_class_summary

,evidence_class,listing_identities,rows_involved,additional_observations
0,confirmed_same_source_exact_content,574,2682,2108
1,confirmed_same_source_reviewed_metadata_conflict,10,78,68
2,confirmed_same_source_strong_path_evidence,777,5758,4981


## 4. Conservative confirmed cross-site evidence

Cross-site confirmation remains independent of Computrabajo repetition. It requires exact canonical URL identity or exact normalized description within non-placeholder company and title context. Exact title-company-location matches are only probable review candidates because recurring openings can share those fields.

In [17]:
fingerprint_context = (
    analysis_jobs["title_norm"].ne("") & analysis_jobs["company_norm"].ne("")
    & ~analysis_jobs["company_placeholder"]
)
location_eligible = fingerprint_context & analysis_jobs["location_norm"].ne("") & ~analysis_jobs["generic_title"]
description_eligible = fingerprint_context & analysis_jobs["description_hash"].ne("")
analysis_jobs = analysis_jobs.assign(location_fingerprint="", description_fingerprint="")
analysis_jobs.loc[location_eligible, "location_fingerprint"] = analysis_jobs.loc[
    location_eligible, ["title_norm", "company_norm", "location_norm"]
].agg("|".join, axis=1)
analysis_jobs.loc[description_eligible, "description_fingerprint"] = analysis_jobs.loc[
    description_eligible, ["title_norm", "company_norm", "description_hash"]
].agg("|".join, axis=1)


def cross_site_spanning_edges(frame: pd.DataFrame, key: str, evidence: str) -> pd.DataFrame:
    """Create deterministic cross-site edges connecting each qualifying exact-key group."""
    records = []
    eligible = frame.loc[frame[key].ne("") & frame["site_norm"].ne("")].sort_values(
        [key, "site_norm", "row_number"]
    )
    for _, group in eligible.groupby(key, sort=True):
        sites = sorted(group["site_norm"].unique())
        if len(sites) < 2:
            continue
        anchor = int(group.loc[group["site_norm"].eq(sites[0]), "row_number"].iloc[0])
        bridge = int(group.loc[group["site_norm"].eq(sites[1]), "row_number"].iloc[0])
        for row in group.itertuples(index=False):
            target = bridge if row.site_norm == sites[0] else anchor
            if int(row.row_number) != target:
                left, right = sorted((int(row.row_number), target))
                records.append({"left": left, "right": right, "evidence": evidence})
    return pd.DataFrame(records, columns=["left", "right", "evidence"]).drop_duplicates()


canonical_edges = cross_site_spanning_edges(analysis_jobs, "canonical_evidence_key", "canonical_url")
description_edges = cross_site_spanning_edges(analysis_jobs, "description_fingerprint", "exact_description_context")
probable_location_edges = cross_site_spanning_edges(analysis_jobs, "location_fingerprint", "title_company_location")
confirmed_cross_site_edges = pd.concat([canonical_edges, description_edges], ignore_index=True).sort_values(
    ["left", "right", "evidence"]
).groupby(["left", "right"], as_index=False, sort=True).agg(
    evidence=("evidence", lambda values: " + ".join(sorted(set(values))))
)


class UnionFind:
    """Build deterministic connected components from confirmed edges."""

    def __init__(self) -> None:
        """Initialize an empty parent mapping."""
        self.parent: dict[int, int] = {}

    def find(self, item: int) -> int:
        """Return the smallest stable root for an item."""
        self.parent.setdefault(item, item)
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, left: int, right: int) -> None:
        """Join two items while keeping the smaller root deterministic."""
        left_root, right_root = self.find(left), self.find(right)
        if left_root != right_root:
            smaller, larger = sorted((left_root, right_root))
            self.parent[larger] = smaller


def components_from_edges(edges: pd.DataFrame) -> tuple[pd.DataFrame, dict[int, str]]:
    """Return component summaries and stable cluster IDs for an edge table."""
    components = UnionFind()
    for edge in edges.itertuples(index=False):
        components.union(int(edge.left), int(edge.right))
    roots = {row: components.find(row) for row in sorted(components.parent)}
    ordered_roots = sorted(set(roots.values()))
    root_ids = {root: f"DUP-{position:05d}" for position, root in enumerate(ordered_roots, 1)}
    row_ids = {row: root_ids[root] for row, root in roots.items()}
    members = pd.DataFrame({"row_number": list(row_ids), "cluster_id": list(row_ids.values())})
    summary = members.groupby("cluster_id", sort=True).size().rename("rows_involved").reset_index()
    summary["excess_rows"] = summary["rows_involved"] - 1
    return summary, row_ids


cross_site_cluster_summary, cross_site_cluster_by_row = components_from_edges(confirmed_cross_site_edges)
analysis_jobs["confirmed_cross_site_cluster_id"] = analysis_jobs["row_number"].map(cross_site_cluster_by_row).astype("string")
confirmed_cross_site_excess = int(cross_site_cluster_summary["excess_rows"].sum())
cross_site_cluster_summary

,cluster_id,rows_involved,excess_rows
0,DUP-00001,2,1
1,DUP-00002,6,5
2,DUP-00003,11,10
3,DUP-00004,8,7
4,DUP-00005,12,11
...,...,...,...
64,DUP-00065,4,3
65,DUP-00066,2,1
66,DUP-00067,2,1
67,DUP-00068,3,2


## 5. Bounded probable and fuzzy review pairs

Only rows outside confirmed cross-site clusters are compared. Blocks use conservative normalized company plus exact normalized location, must span sites, and are capped by `MAX_FUZZY_BLOCK_ROWS`. Candidates never enter a confirmed union.

In [18]:
def jaccard_tokens(left: str, right: str) -> float:
    """Return Jaccard similarity for two whitespace-token sets."""
    left_tokens, right_tokens = set(left.split()), set(right.split())
    union = left_tokens | right_tokens
    return len(left_tokens & right_tokens) / len(union) if union else 0.0


confirmed_cross_site_mask = analysis_jobs["confirmed_cross_site_cluster_id"].notna()
fuzzy_eligible = (
    ~confirmed_cross_site_mask & analysis_jobs["company_norm"].ne("")
    & ~analysis_jobs["company_placeholder"] & analysis_jobs["site_norm"].ne("")
    & analysis_jobs["title_norm"].ne("") & ~analysis_jobs["generic_title"]
)
location_block = analysis_jobs["location_norm"].mask(analysis_jobs["location_norm"].eq(""), "<missing>")
analysis_jobs["fuzzy_block_key"] = analysis_jobs["company_norm"] + "|" + location_block
block_stats = analysis_jobs.loc[fuzzy_eligible].groupby("fuzzy_block_key", sort=True).agg(
    block_rows=("row_number", "size"), source_count=("site_norm", "nunique"),
).reset_index()
eligible_blocks = set(block_stats.loc[
    block_stats["source_count"].gt(1) & block_stats["block_rows"].le(MAX_FUZZY_BLOCK_ROWS), "fuzzy_block_key",
])
fuzzy_population = analysis_jobs.loc[
    fuzzy_eligible & analysis_jobs["fuzzy_block_key"].isin(eligible_blocks)
].sort_values(["fuzzy_block_key", "site_norm", "row_number"])
candidate_records = []
for _, block in fuzzy_population.groupby("fuzzy_block_key", sort=True):
    for left, right in combinations(list(block.itertuples(index=False)), 2):
        if left.site_norm == right.site_norm:
            continue
        title_jaccard = jaccard_tokens(left.title_norm, right.title_norm)
        exact_location = bool(left.location_norm) and left.location_norm == right.location_norm
        exact_description = bool(left.description_hash) and left.description_hash == right.description_hash
        if title_jaccard < FUZZY_MIN_TITLE_JACCARD or (not exact_location and not exact_description):
            continue
        candidate_records.append({
            "left_row": int(left.row_number), "right_row": int(right.row_number),
            "left_site": left.site, "right_site": right.site, "company_norm": left.company_norm,
            "title_jaccard": round(title_jaccard, 4), "exact_location": exact_location,
            "exact_description_hash": exact_description,
            "candidate_type": ("probable_exact_title_company_location"
                if title_jaccard == 1.0 and exact_location else "fuzzy_review"),
        })
fuzzy_candidates = pd.DataFrame(candidate_records, columns=[
    "left_row", "right_row", "left_site", "right_site", "company_norm",
    "title_jaccard", "exact_location", "exact_description_hash", "candidate_type",
]).sort_values(["candidate_type", "title_jaccard", "left_row", "right_row"], ascending=[True, False, True, True])
raw_review_fields = ["title", "company", "raw_location", "job_url", "date_scraped"]
left_raw = analysis_jobs.set_index("row_number")[raw_review_fields].add_prefix("left_")
right_raw = analysis_jobs.set_index("row_number")[raw_review_fields].add_prefix("right_")
fuzzy_candidates.head(SAMPLE_ROWS).join(left_raw, on="left_row").join(right_raw, on="right_row")

,left_row,right_row,left_site,right_site,company_norm,title_jaccard,exact_location,exact_description_hash,candidate_type,left_title,left_company,left_raw_location,left_job_url,left_date_scraped,right_title,right_company,right_raw_location,right_job_url,right_date_scraped
209,4427,4551,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:20:42.887782,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21027051/,2026-05-10 08:22:05.516822
210,4427,19408,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:20:42.887782,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21223432/,2026-06-28 14:20:12.984931
211,4428,5224,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,CYBERSECURITY IAM ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:20:42.887782,CYBERSECURITY IAM ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21027120/,2026-05-10 14:02:00.803355
212,4428,11014,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,CYBERSECURITY IAM ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:20:42.887782,CYBERSECURITY IAM ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21120954/,2026-05-25 15:37:28.114132
213,4429,5249,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,CYBERSECURITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:20:42.887782,CYBERSECURITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21027135/,2026-05-10 14:02:00.803355
214,4429,11013,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,CYBERSECURITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:20:42.887782,CYBERSECURITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21120953/,2026-05-25 15:37:28.114132
215,4504,4551,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:21:13.901641,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21027051/,2026-05-10 08:22:05.516822
216,4504,19408,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:21:13.901641,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21223432/,2026-06-28 14:20:12.984931
217,4548,4551,computrabajo,occ,nov shared services mexico,1.0,True,False,probable_exact_title_company_location,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://mx.computrabajo.com/ofertas-de-trabajo...,2026-05-10 08:21:58.603637,SITE RELIABILITY ENGINEER,NOV SHARED SERVICES MEXICO,"Monterrey, Nuevo León",https://www.occ.com.mx/empleo/oferta/21027051/,2026-05-10 08:2

## 6. True confirmed union

The combined distinct-vacancy estimate is computed from one graph containing automatically confirmed Computrabajo groups, the ten integrity-checked reviewed groups, and conservative confirmed cross-site edges. The graph union, not arithmetic addition, handles overlap. Future unresolved ambiguous groups and probable/fuzzy pairs never become confirmed edges.

In [19]:
confirmed_source_groups = computrabajo_listing_level.loc[
    computrabajo_listing_level["evidence_class"].ne("ambiguous_or_conflicting_source_observations")
].copy()
confirmed_source_evidence = confirmed_source_groups.set_index("source_listing_identity")["evidence_class"]
confirmed_source_identities = set(confirmed_source_groups["source_listing_identity"])
same_source_edge_records = []
for identity, group in computrabajo_repeated_rows.loc[
    computrabajo_repeated_rows["source_listing_identity"].isin(confirmed_source_identities)
].groupby("source_listing_identity", sort=True):
    representative = int(representatives.loc[identity, "row_number"])
    for row_number in sorted(group["row_number"].astype(int)):
        if row_number != representative:
            left, right = sorted((representative, row_number))
            same_source_edge_records.append({
                "left": left, "right": right, "evidence": confirmed_source_evidence.loc[identity],
            })
same_source_edges = pd.DataFrame(
    same_source_edge_records, columns=["left", "right", "evidence"]
).drop_duplicates()
confirmed_union_edges = pd.concat(
    [same_source_edges, confirmed_cross_site_edges], ignore_index=True
).sort_values(["left", "right", "evidence"]).groupby(
    ["left", "right"], as_index=False, sort=True
).agg(evidence=("evidence", lambda values: " + ".join(sorted(set(values)))))
confirmed_union_summary, confirmed_union_cluster_by_row = components_from_edges(confirmed_union_edges)
analysis_jobs["confirmed_union_cluster_id"] = analysis_jobs["row_number"].map(confirmed_union_cluster_by_row).astype("string")
confirmed_source_excess = int(confirmed_source_groups["additional_observations"].sum())
computrabajo_source_identity_count = len(computrabajo_rows) - confirmed_source_excess
confirmed_union_excess = int(confirmed_union_summary["excess_rows"].sum())
combined_distinct_vacancies = len(analysis_jobs) - confirmed_union_excess
overlap_avoided = confirmed_source_excess + confirmed_cross_site_excess - confirmed_union_excess
singleton_keys = analysis_jobs["row_number"].map(lambda row: f"ROW-{int(row):08d}").astype("string")
analysis_jobs["distinct_vacancy_key"] = analysis_jobs["confirmed_union_cluster_id"].fillna(singleton_keys)
left_union_clusters = fuzzy_candidates["left_row"].map(confirmed_union_cluster_by_row).astype("string")
right_union_clusters = fuzzy_candidates["right_row"].map(confirmed_union_cluster_by_row).astype("string")
same_confirmed_union_component = (
    left_union_clusters.notna() & right_union_clusters.notna()
    & left_union_clusters.eq(right_union_clusters).fillna(False)
)
fuzzy_candidate_union_diagnostics = fuzzy_candidates.assign(
    left_confirmed_union_cluster_id=left_union_clusters,
    right_confirmed_union_cluster_id=right_union_clusters,
    redundant_after_confirmed_union=same_confirmed_union_component,
)
redundant_fuzzy_candidates = fuzzy_candidate_union_diagnostics.loc[
    fuzzy_candidate_union_diagnostics["redundant_after_confirmed_union"]
].copy()
unresolved_fuzzy_candidates = fuzzy_candidate_union_diagnostics.loc[
    ~fuzzy_candidate_union_diagnostics["redundant_after_confirmed_union"]
].copy()

## 7. Final metrics

Reviewed confirmed groups and future unresolved ambiguous groups are reported separately. Computrabajo confirmed excess and source identities are derived from the confirmed groups. The combined count remains the true graph union and is never constructed by arithmetically adding evidence categories.

In [20]:
raw_url_excess = int(key_audit.loc[key_audit["key"].eq("job_url"), "excess_copies"].iloc[0])
final_metrics = pd.DataFrame([
    {"category": "exact raw URL duplicate excess", "value": raw_url_excess, "unit": "rows"},
    {"category": "Computrabajo confirmed repeated observations", "value": confirmed_source_excess, "unit": "excess rows"},
    {"category": "Computrabajo source identities after confirmed union", "value": computrabajo_source_identity_count, "unit": "identities"},
    {"category": "reviewed confirmed Computrabajo metadata-conflict groups", "value": len(reviewed_metadata_conflict_groups), "unit": "groups"},
    {"category": "reviewed confirmed Computrabajo metadata-conflict rows", "value": int(reviewed_metadata_conflict_groups["times_seen"].sum()), "unit": "rows"},
    {"category": "future unresolved ambiguous Computrabajo groups", "value": len(unresolved_ambiguous_groups), "unit": "groups"},
    {"category": "future unresolved ambiguous Computrabajo rows", "value": int(unresolved_ambiguous_groups["times_seen"].sum()), "unit": "rows"},
    {"category": "conservative confirmed cross-site excess", "value": confirmed_cross_site_excess, "unit": "rows"},
    {"category": "probable/fuzzy unresolved manual-review pairs", "value": len(unresolved_fuzzy_candidates), "unit": "pairs"},
    {"category": "fuzzy pairs redundant after confirmed union", "value": len(redundant_fuzzy_candidates), "unit": "pairs"},
    {"category": "combined confirmed union excess", "value": confirmed_union_excess, "unit": "rows"},
    {"category": "combined confirmed distinct vacancies", "value": combined_distinct_vacancies, "unit": "vacancies"},
    {"category": "arithmetic double-counting avoided by union", "value": overlap_avoided, "unit": "rows"},
])
final_metrics

,category,value,unit
0,exact raw URL duplicate excess,0,rows
1,Computrabajo confirmed repeated observations,7157,excess rows
2,Computrabajo source identities after confirmed...,2304,identities
3,reviewed confirmed Computrabajo metadata-confl...,10,groups
4,reviewed confirmed Computrabajo metadata-confl...,78,rows
5,future unresolved ambiguous Computrabajo groups,0,groups
6,future unresolved ambiguous Computrabajo rows,0,rows
7,conservative confirmed cross-site excess,420,rows
8,probable/fuzzy unresolved manual-review pairs,348,pairs
9,fuzzy pairs redundant after confirmed union,0,pairs


## 8. Validation assertions

These assertions enforce the approved-ID match, promotion count, reviewed-row partition, complete decisions, path evidence, graph union, and continued isolation of future unresolved groups.

In [21]:
assert len(analysis_jobs) == EXPECTED_ROWS
assert analysis_jobs["row_number"].is_unique
assert approved_identifiers_found == REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS
assert len(reviewed_metadata_conflict_groups) == EXPECTED_REVIEWED_METADATA_CONFLICT_GROUPS
assert reviewed_metadata_conflict_groups["times_seen"].sum() == EXPECTED_REVIEWED_METADATA_CONFLICT_ROWS
assert reviewed_metadata_conflict_groups["evidence_class"].eq(REVIEWED_METADATA_CONFLICT_EVIDENCE_CLASS).all()
assert reviewed_metadata_conflict_groups["automatic_evidence_class"].eq(
    "ambiguous_or_conflicting_source_observations"
).all()
assert set(reviewed_metadata_conflict_groups["path_identifier"]) == REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS
assert len(approved_underlying_rows) == EXPECTED_REVIEWED_METADATA_CONFLICT_ROWS
assert set(approved_underlying_rows["source_listing_identity"]) == set(
    reviewed_metadata_conflict_groups["source_listing_identity"]
)
assert not set(unresolved_ambiguous_groups["path_identifier"]) & REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS
assert set(unresolved_ambiguous_groups["source_listing_identity"]).isdisjoint(confirmed_source_identities)
assert set(REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS) == set(REVIEWED_METADATA_CONFLICT_RATIONALE_BY_PATH_IDENTIFIER)
assert confirmed_source_excess == 7_157
assert computrabajo_source_identity_count == 2_304
assert computrabajo_listing_level["times_seen"].sum() == len(computrabajo_repeated_rows)
assert computrabajo_repeated_rows.groupby("source_listing_identity")["canonical_path"].nunique().eq(1).all()
assert confirmed_source_groups["fragment_free_url_count"].eq(1).all()
assert confirmed_source_groups["expected_fragment_rows"].eq(confirmed_source_groups["times_seen"]).all()
assert representatives.index.is_unique
assert set(representatives.index) == repeated_identities
assert same_source_edges[["left", "right"]].duplicated().sum() == 0
assert confirmed_cross_site_edges[["left", "right"]].duplicated().sum() == 0
assert confirmed_union_edges[["left", "right"]].duplicated().sum() == 0
reviewed_row_numbers = set(approved_underlying_rows["row_number"].astype(int))
reviewed_edge_rows = set(same_source_edges.loc[
    same_source_edges["evidence"].eq(REVIEWED_METADATA_CONFLICT_EVIDENCE_CLASS), ["left", "right"]
].stack().astype(int))
assert reviewed_row_numbers == reviewed_edge_rows
reviewed_edge_pairs = set(map(tuple, same_source_edges.loc[
    same_source_edges["evidence"].eq(REVIEWED_METADATA_CONFLICT_EVIDENCE_CLASS), ["left", "right"]
].to_numpy()))
confirmed_union_pairs = set(map(tuple, confirmed_union_edges[["left", "right"]].to_numpy()))
assert reviewed_edge_pairs <= confirmed_union_pairs
assert confirmed_union_excess == int(confirmed_union_summary["rows_involved"].sum() - len(confirmed_union_summary))
assert combined_distinct_vacancies + confirmed_union_excess == len(analysis_jobs)
assert confirmed_union_excess <= confirmed_source_excess + confirmed_cross_site_excess
assert overlap_avoided >= 0
assert analysis_jobs["distinct_vacancy_key"].notna().all()
assert analysis_jobs["distinct_vacancy_key"].nunique() == combined_distinct_vacancies
assert len(fuzzy_candidates) == len(unresolved_fuzzy_candidates) + len(redundant_fuzzy_candidates)

## 9. Reviewed Computrabajo metadata-conflict decision audit

The user approved all ten reviewed path-identity groups as `same_vacancy`. The audit below preserves the 10-group summary and all 78 underlying rows, records concise evidence-derived rationale categories, and separately exposes any future ambiguous groups that were not part of this approval.

In [22]:
def readable_distinct_values(values: pd.Series) -> str:
    """Return deterministic, reviewable distinct non-empty values."""
    distinct_values = {str(value).strip() for value in values.dropna() if str(value).strip()}
    return " | ".join(sorted(distinct_values, key=lambda value: (normalize_text(value), value)))


reviewed_identity_order = sorted(reviewed_metadata_conflict_groups["source_listing_identity"].astype(str))
reviewed_identity_set = set(reviewed_identity_order)
reviewed_review_rows = computrabajo_repeated_rows.loc[
    computrabajo_repeated_rows["source_listing_identity"].isin(reviewed_identity_set)
].copy()
reviewed_context_summary = reviewed_review_rows.groupby("source_listing_identity", sort=True).agg(
    distinct_titles=("title", readable_distinct_values),
    distinct_companies=("company", readable_distinct_values),
).reset_index()
reviewed_decision_audit = reviewed_metadata_conflict_groups.loc[:, [
    "source_listing_identity", "canonical_path", "path_identifier", "times_seen",
    "additional_observations", "first_seen", "last_seen",
    "unique_nonempty_descriptions", "descriptions_present",
]].rename(columns={
    "times_seen": "rows_involved", "additional_observations": "confirmed_excess_rows",
    "unique_nonempty_descriptions": "description_variants",
}).merge(reviewed_context_summary, on="source_listing_identity", how="left", validate="one_to_one")
reviewed_decision_audit["descriptions_missing"] = (
    reviewed_decision_audit["rows_involved"] - reviewed_decision_audit["descriptions_present"]
)
reviewed_decision_audit["reviewer_label"] = "same_vacancy"
reviewed_decision_audit["rationale_category"] = reviewed_decision_audit["path_identifier"].map(
    REVIEWED_METADATA_CONFLICT_RATIONALE_BY_PATH_IDENTIFIER
).astype("string")
reviewed_decision_audit["decision_provenance"] = "user-approved analytical decision"
reviewed_decision_audit = reviewed_decision_audit.sort_values(
    "source_listing_identity", kind="mergesort"
).reset_index(drop=True)
description_text = reviewed_review_rows["description"].fillna("").astype("string").str.replace(
    r"\s+", " ", regex=True
).str.strip()
description_preview = description_text.map(lambda value: (
    value[:DESCRIPTION_PREVIEW_MAX_CHARACTERS - 3] + "..."
    if len(value) > DESCRIPTION_PREVIEW_MAX_CHARACTERS else value
))
reviewed_review_detail = reviewed_review_rows.assign(description_preview=description_preview).sort_values(
    ["source_listing_identity", "date_scraped_parsed", "stable_id_tiebreaker", "row_number"],
    kind="mergesort",
).loc[:, [
    "source_listing_identity", "id", "date_scraped_parsed", "title", "company",
    "job_url", "raw_fragment", "expected_tracking_fragment", "description_preview",
]].rename(columns={"id": "original_row_id", "date_scraped_parsed": "date_scraped"}).reset_index(drop=True)
unresolved_ambiguous_review_table = unresolved_ambiguous_groups.loc[:, [
    "source_listing_identity", "canonical_path", "path_identifier", "times_seen",
    "additional_observations", "automatic_evidence_class",
]].rename(columns={
    "times_seen": "rows_involved", "additional_observations": "potential_excess_rows",
}).sort_values("source_listing_identity", kind="mergesort").reset_index(drop=True)
assert len(reviewed_decision_audit) == EXPECTED_REVIEWED_METADATA_CONFLICT_GROUPS
assert reviewed_decision_audit["rows_involved"].sum() == EXPECTED_REVIEWED_METADATA_CONFLICT_ROWS
assert len(reviewed_review_detail) == EXPECTED_REVIEWED_METADATA_CONFLICT_ROWS
assert reviewed_decision_audit["reviewer_label"].eq("same_vacancy").all()
assert reviewed_decision_audit["rationale_category"].notna().all()
assert reviewed_decision_audit["decision_provenance"].eq("user-approved analytical decision").all()
assert set(reviewed_decision_audit["path_identifier"]) == REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS
assert set(unresolved_ambiguous_review_table["path_identifier"]).isdisjoint(
    REVIEWED_METADATA_CONFLICT_PATH_IDENTIFIERS
)
{
    "reviewed_decisions": reviewed_decision_audit,
    "reviewed_underlying_rows": reviewed_review_detail,
    "future_unresolved_groups": unresolved_ambiguous_review_table,
}

{'reviewed_decisions':                              source_listing_identity  \
 0  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 1  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 2  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 3  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 4  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 5  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 6  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 7  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 8  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 9  computrabajo:mx.computrabajo.com/ofertas-de-tr...   
 
                                       canonical_path  \
 0  /ofertas-de-trabajo/oferta-de-trabajo-de-anali...   
 1  /ofertas-de-trabajo/oferta-de-trabajo-de-anali...   
 2  /ofertas-de-trabajo/oferta-de-trabajo-de-anali...   
 3  /ofertas-de-trabajo/oferta-de-trabajo-de-anali...   
 4  /ofertas-de-trabajo/oferta-de-trabajo-de-anali...   
 5  /of

## 10. Interpretation and downstream use

Computrabajo is the only platform in this snapshot with a large, directly demonstrated same-platform pattern where the vacancy path stays exact and only the `#lc=ListOffers-Score...` fragment changes. Automatic exact-content and strong-path classifications are unchanged. The ten reviewed metadata-conflict groups enter the same-source graph only through the explicit user-approved identifier set and only while their path and fragment evidence remains intact.

After rerun, the expected Computrabajo source-identity count is 2,304 and its confirmed excess is derived as 7,157. The final combined union count must be read from the executed graph result; it is intentionally not claimed in this unexecuted notebook source. Any future ambiguous Computrabajo groups remain separate in `unresolved_ambiguous_review_table`.

## Next step

Rerun the notebook from the beginning, verify the reviewed-decision audit contains 10 groups and 78 rows, and inspect the derived true-union metrics. Review any entries in `unresolved_ambiguous_review_table` independently; do not extend the approved decision to new path identifiers.